# 14. 다른 모델 계열 비교 + 확률 섞기 (H14)

## 배경
- 최고 LB: 10 XGB (0.3206, 정규화 전)
- 전처리 변경: `load_data()`에 변이 표기 정규화 추가 → 10 설정 CV 0.4113 → 0.4161 (15에서 확인)
- 13·13b: 가짜 변이에 대한 강건성은 LR(stats 없음)과 XGB(stats 있음)가 반대로 나타남
  - 13b 대조군 결과, 이 차이는 모델 구조가 아니라 stats 유무 때문 → "약점이 정반대"는 섞기의 근거로 쓰지 않음
- 그래도 성격이 다른 모델은 OOF에서 예측이 서로 다를 수 있음 → 섞기 효과는 OOF와 LB로만 판단

## 가설
- H14: 성격이 다른 모델(LR, 배깅 트리, 부스팅 트리)의 확률을 섞으면 XGB 단독보다 LB가 오른다

## 검증 방법
- 데이터: `load_data()` (정규화, dedupe=True) → 10 + 정규화(15)와 같은 전처리
- 피처: LR은 any(05와 동일), 트리 모델은 any + stats
- 모델: LR(05), XGB(10), RandomForest, ExtraTrees, LightGBM
- 모든 모델의 OOF 확률과 test 확률을 한 번만 계산해 저장 → 섞는 조합은 저장값으로 빠르게 비교
- 섞는 비율은 OOF로만 고름 (test 정보 미사용, 예측 비율을 train 비율로 맞추는 후처리는 하지 않음)
- OOF로 고른 비율은 낙관적일 수 있음 → 단순한 조합(같은 비율 평균)을 함께 제출 후보로 둠

#### 준비
- 피처 세트 2종: any(LR용), any + stats(트리용)
- 모든 모델이 같은 fold로 평가 → OOF 확률을 서로 섞을 수 있음

In [2]:
import sys; sys.path.append('..')
import time, itertools, numpy as np, pandas as pd, xgboost as xgb, lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, macro_f1, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
X_any,  X_any_test  = F_tr['any'].astype('float32'), F_te['any'].astype('float32')
X_tree, X_tree_test = pd.concat([F_tr['any'], F_tr['stats']], axis=1), pd.concat([F_te['any'], F_te['stats']], axis=1)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

#### 비교할 모델
- LR, XGB: 05·10과 같은 설정 (LB 확인된 모델)
- RandomForest, ExtraTrees: 깊은 나무를 각자 키워 평균내는 배깅 방식 → XGB(부스팅)와 성격이 가장 다름
  - `balanced_subsample`: 나무마다 뽑힌 데이터 기준으로 클래스 가중치 계산
  - `min_samples_leaf=2`: 잎에 최소 2명 → 한 명을 외우는 것 방지
- LightGBM: 10에서 효과 있었던 얕은 구조 + 피처 샘플링 0.3을 비슷하게 적용

In [3]:
MODELS = {   # 이름: (모델 만드는 함수, train 피처, test 피처)
    'LR': (lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
           X_any, X_any_test),
    'XGB': (lambda: BalancedXGB(n_estimators=200, learning_rate=0.1, max_depth=3, colsample_bytree=0.3,
                                tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1),
            X_tree, X_tree_test),
    'RF': (lambda: RandomForestClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2,
                                          class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
           X_tree, X_tree_test),
    'ET': (lambda: ExtraTreesClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2,
                                        class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
           X_tree, X_tree_test),
    'LGBM': (lambda: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                                        colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
                                        min_child_samples=10, class_weight='balanced',
                                        random_state=SEED, n_jobs=-1, verbose=-1),
             X_tree, X_tree_test),
}

#### 모델별 CV + 확률 저장
- `run_cv`가 돌려주는 OOF 확률(train 각 환자를 못 본 모델의 예측)과 test 확률(fold 모델 5개 평균)을 저장
- 이후 섞기는 저장값만 사용 → 다시 학습하지 않음

In [4]:
P_oof, P_test, rows = {}, {}, []
for name, (mk, Xtr, Xte) in MODELS.items():
    t = time.time()
    r = run_cv(mk, Xtr, y, Xte, verbose=False)
    P_oof[name], P_test[name] = r['oof_proba'], r['test_proba']
    rows.append(dict(model=name, cv=round(r['cv_mean'], 4), std=round(r['cv_std'], 4),
                     oof=round(r['oof_score'], 4), sec=round(time.time() - t)))
    print(rows[-1])
res14 = pd.DataFrame(rows).sort_values('cv', ascending=False)
res14

{'model': 'LR', 'cv': 0.3818, 'std': 0.0064, 'oof': 0.383, 'sec': 7}
{'model': 'XGB', 'cv': 0.4161, 'std': 0.006, 'oof': 0.4167, 'sec': 151}
{'model': 'RF', 'cv': 0.3892, 'std': 0.0093, 'oof': 0.389, 'sec': 25}
{'model': 'ET', 'cv': 0.3793, 'std': 0.0152, 'oof': 0.38, 'sec': 58}
{'model': 'LGBM', 'cv': 0.3892, 'std': 0.0071, 'oof': 0.3908, 'sec': 80}


,model,cv,std,oof,sec
1,XGB,0.4161,0.0060,0.4167,151
4,LGBM,0.3892,0.0071,0.3908,80
2,RF,0.3892,0.0093,0.3890,25
0,LR,0.3818,0.0064,0.3830,7
3,ET,0.3793,0.0152,0.3800,58


#### 모델 간 예측 불일치율 (OOF)
- 두 모델의 예측이 다른 환자 비율 → 높을수록 섞었을 때 보완 효과를 기대할 수 있음
- 너무 비슷한 모델끼리는 섞어도 효과가 작음

In [5]:
pred = {k: v.argmax(1) for k, v in P_oof.items()}
names = list(pred)
dis = pd.DataFrame([[np.mean(pred[a] != pred[b]) for b in names] for a in names], index=names, columns=names)
dis.round(3)

,LR,XGB,RF,ET,LGBM
LR,0.000,0.422,0.386,0.339,0.445
XGB,0.422,0.000,0.307,0.329,0.367
RF,0.386,0.307,0.000,0.182,0.403
ET,0.339,0.329,0.182,0.000,0.398
LGBM,0.445,0.367,0.403,0.398,0.000


#### XGB + 다른 모델 섞기 (OOF 기준)
- XGB를 중심으로 다른 모델 하나를 섞는 비율을 0~0.5에서 비교
- 확률 크기를 맞추려고 섞기 전에 각 모델 확률을 행 합 1로 정규화 (LR·트리 모두 이미 1이지만 안전장치)
- 셋 이상 섞는 조합은 같은 비율 평균만 비교 (비율까지 맞추면 OOF 과적합 위험)

In [6]:
def blend_score(weights):
    P = sum(w * P_oof[k] for k, w in weights.items())
    return macro_f1(y, P.argmax(1))

rows = [dict(blend='XGB 단독', score=round(blend_score({'XGB': 1}), 4))]
for other in ['LR', 'RF', 'ET', 'LGBM']:
    for w in [0.2, 0.3, 0.4, 0.5]:
        rows.append(dict(blend=f'XGB {1-w:.1f} + {other} {w:.1f}',
                         score=round(blend_score({'XGB': 1 - w, other: w}), 4)))
for combo in [('XGB', 'LR', 'RF'), ('XGB', 'LR', 'ET'), ('XGB', 'LR', 'RF', 'ET'), ('XGB', 'LR', 'RF', 'ET', 'LGBM')]:
    rows.append(dict(blend=' + '.join(combo) + ' (같은 비율)',
                     score=round(blend_score({k: 1 / len(combo) for k in combo}), 4)))
blend_res = pd.DataFrame(rows).sort_values('score', ascending=False)
blend_res.head(12)

,blend,score
17,XGB + LR + RF (같은 비율),0.4246
19,XGB + LR + RF + ET (같은 비율),0.4241
18,XGB + LR + ET (같은 비율),0.4231
2,XGB 0.7 + LR 0.3,0.4231
3,XGB 0.6 + LR 0.4,0.4221
1,XGB 0.8 + LR 0.2,0.4215
20,XGB + LR + RF + ET + LGBM (같은 비율),0.4210
4,XGB 0.5 + LR 0.5,0.4206
12,XGB 0.5 + ET 0.5,0.4203
11,XGB 0.6 + ET 0.4,0.4201


#### 제출 파일
- 셀 6에서 고른 조합을 test 확률로 똑같이 섞어서 저장
- OOF 점수 차이가 작으면 더 단순한 조합(모델 수가 적거나 같은 비율)을 우선

In [7]:
CHOSEN = {'XGB': 0.7, 'LR': 0.3}   # 정규화 후 XGB+LR 조합 중 OOF 최고
P = sum(w * P_test[k] for k, w in CHOSEN.items())
save_submission(submission, P, le, name='blend_XGB0.7_LR0.3', cv_score=blend_score(CHOSEN))

CHOSEN = {k: 0.25 for k in ['XGB', 'LR', 'RF', 'ET']}
P = sum(w * P_test[k] for k, w in CHOSEN.items())
save_submission(submission, P, le, name='blend_XGB_LR_RF_ET_equal', cv_score=blend_score(CHOSEN))

저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1111_blend_XGB0.7_LR0.3_0.423.csv
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1111_blend_XGB_LR_RF_ET_equal_0.424.csv


WindowsPath('C:/Users/USER/Desktop/hackathon/submissions/1111_blend_XGB_LR_RF_ET_equal_0.424.csv')

## 결과

전처리 변경(변이 표기 정규화, `load_data()` 기본값) 후 다시 실행한 결과입니다.

**모델 단독**

| 모델 | CV (± std) | OOF |
| :--- | :-: | :-: |
| XGB (10 설정) | 0.4161 ± 0.0060 | 0.4167 |
| LightGBM | 0.3892 ± 0.0071 | 0.3908 |
| RandomForest | 0.3892 ± 0.0093 | 0.3890 |
| LR (05 설정, any) | 0.3818 ± 0.0064 | 0.3830 |
| ExtraTrees | 0.3793 ± 0.0152 | 0.3800 |

- XGB 0.4161 ± 0.0060은 15(any + stats)와 동일 → 같은 조건 재현 확인
- LR은 any만 써서 정규화 영향 없음 (05와 동일)

**XGB와의 예측 불일치율 (OOF)**: LR 0.422 > LGBM 0.367 > ET 0.329 > RF 0.307
- XGB와 가장 다르게 틀리는 모델은 LR → 섞기 재료로 가장 유망
- RF와 ET는 서로 불일치율 0.182로 너무 비슷함 → 둘 다 넣을 필요 적음

**섞기 (OOF Macro F1, XGB 단독 0.4167)**

| 조합 | OOF | XGB 단독 대비 |
| :--- | :-: | :-: |
| XGB + LR + RF (같은 비율) | 0.4246 | +0.008 |
| XGB + LR + RF + ET (같은 비율) | 0.4241 | +0.007 |
| **XGB 0.7 + LR 0.3** | **0.4231** | **+0.006** |
| XGB 0.6 + LR 0.4 | 0.4221 | +0.005 |
| XGB + LR + RF + ET + LGBM (같은 비율) | 0.4210 | +0.004 |

- 상위 조합에 모두 LR이 들어 있음 → 섞기 효과는 대부분 LR에서 나옴
- LGBM을 넣으면 하락 (XGB와 같은 부스팅 계열이라 새 정보가 적고 단독 성능도 약함)
- 섞기 개선 폭(+0.006~0.008)은 XGB 단독 std(0.006)와 비슷한 크기 → LB로 확인 필요

### 참고: 정규화 전 결과

| 조합 | OOF |
| :--- | :-: |
| XGB 단독 | 0.4120 |
| XGB 0.6 + LR 0.4 | 0.4228 (+0.011) |
| XGB + LR + RF + ET (같은 비율) | 0.4246 (+0.013) |

- 정규화로 XGB가 오르면서(0.4120 → 0.4167) LR이 보태주는 몫은 +0.011 → +0.006으로 줄어듦

**제출 후보**
1. XGB 0.7 + LR 0.3: 모델 2개로 가장 단순, "LR을 섞으면 LB가 오르나"를 깔끔하게 확인 → LB (제출 후 기입)
2. XGB + LR + RF (같은 비율): OOF 1위, 비율을 맞추지 않은 같은 비율 평균이라 OOF 과적합 위험이 적음 → 예비

## 결론
- H14 채택 / 기각: (LB 비교 후 기입, 비교 대상은 같은 전처리의 "10 + 정규화")
- 13b 결과대로 섞기의 근거("약점이 정반대")는 내려놓고, 효과는 OOF와 LB로만 판단